# Two-sample and paired t-tests
Desktop against mobile users, Titanic male against female ages, a weight-loss program, and two models compared on the same cross-validation folds.
Note: website_time.csv holds simulated values built to match the summary numbers of the example.

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

## 1. Desktop and mobile users: independent two-sample t-test

In [ ]:
web = pd.read_csv("data/website_time.csv")
desktop = web.loc[web.device == "desktop", "minutes"]
mobile = web.loc[web.device == "mobile", "minutes"]
print(web.groupby("device").minutes.agg(["count", "mean", "std"]).round(2))
# assumptions: normality of each group, equal variances
print("Shapiro p:", round(stats.shapiro(desktop).pvalue, 2), round(stats.shapiro(mobile).pvalue, 2))
print("Levene p: ", round(stats.levene(desktop, mobile).pvalue, 2))

In [ ]:
# the formula by hand
se = np.sqrt(3.5 ** 2 / 30 + 2.7 ** 2 / 30)
t = (18.5 - 14.3) / se
print(f"SE = {se:.3f}, t = {t:.2f}, p = {2 * stats.t.sf(t, df=58):.2g}")
# scipy: Student's test (equal variances) and Welch's test
print(stats.ttest_ind(desktop, mobile))
print(stats.ttest_ind(desktop, mobile, equal_var=False))

## 2. Titanic: are men older than women?

In [ ]:
titanic = pd.concat([pd.read_csv("data/titanic_train.csv"), pd.read_csv("data/titanic_test.csv")])
male = titanic.loc[titanic.Sex == "male", "Age"].dropna().reset_index(drop=True)
female = titanic.loc[titanic.Sex == "female", "Age"].dropna().reset_index(drop=True)
print(len(male), "men,", len(female), "women with known age")
# one random sample of 25 from each group
s_male, s_female = male.sample(25, random_state=5), female.sample(25, random_state=5)
print(f"sample means {s_male.mean():.1f} and {s_female.mean():.1f}")
print("Shapiro p:", round(stats.shapiro(s_male).pvalue, 2), round(stats.shapiro(s_female).pvalue, 2))
print("Levene p: ", round(stats.levene(s_male, s_female).pvalue, 2))
res = stats.ttest_ind(s_male, s_female, alternative="greater")
print(f"t = {res.statistic:.2f}, p = {res.pvalue:.2f}")
# the truth: population means
print(f"population means: men {male.mean():.2f}, women {female.mean():.2f}")

In [ ]:
# power: how often does a pair of samples of 25 detect the real difference?
rej = [stats.ttest_ind(male.sample(25, random_state=i), female.sample(25, random_state=10_000 + i),
                       alternative="greater").pvalue <= 0.05 for i in range(4000)]
print("share of sample pairs that reject H0:", np.mean(rej))

## 3. Weight-loss program: paired t-test

In [ ]:
w = pd.read_csv("data/weight_loss.csv")
d = w["before"] - w["after"]
print("differences:", d.tolist())
print(f"mean {d.mean():.2f}, sd {d.std():.2f}, Shapiro p {stats.shapiro(d).pvalue:.2f}")
# H1: the program reduces weight, i.e. before - after > 0
res = stats.ttest_rel(w["before"], w["after"], alternative="greater")
print(f"t = {res.statistic:.2f}, p = {res.pvalue:.2f}")
# the same as a one-sample t-test of the differences against 0
print(stats.ttest_1samp(d, 0, alternative="greater"))

## 4. Why pairing helps

In [ ]:
se_paired = d.std() / np.sqrt(len(d))
se_indep = np.sqrt(w["before"].var() / 15 + w["after"].var() / 15)
print(f"paired SE {se_paired:.2f}, independent SE {se_indep:.2f}")
# suppose everyone ended 2 kg lighter than recorded
after2 = w["after"] - 2
print("paired:     ", stats.ttest_rel(w["before"], after2, alternative="greater"))
print("independent:", stats.ttest_ind(w["before"], after2, alternative="greater"))

## 5. Comparing two models on the same folds

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

X, y = load_breast_cancer(return_X_y=True)
# the same 10 folds for both models, so the scores are paired
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=0)
acc_lr = cross_val_score(make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000)), X, y, cv=cv)
acc_dt = cross_val_score(DecisionTreeClassifier(random_state=0), X, y, cv=cv)
print(f"mean accuracy: logistic {acc_lr.mean():.3f}, tree {acc_dt.mean():.3f}")
print(stats.ttest_rel(acc_lr, acc_dt))

In [ ]:
# corrected resampled t-test (Nadeau and Bengio): inflate the variance for overlapping training sets
diff = acc_lr - acc_dt
k = len(diff)
t_corr = diff.mean() / np.sqrt((1 / k + 1 / (k - 1)) * diff.var(ddof=1))
print(f"corrected t = {t_corr:.2f}, p = {2 * stats.t.sf(abs(t_corr), df=k - 1):.3f}")